<a href="https://colab.research.google.com/github/milvus-io/bootcamp/blob/master/integration/ag2/ag2_multiagent_rag_with_milvus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>   <a href="https://github.com/milvus-io/bootcamp/blob/master/integration/ag2/ag2_multiagent_rag_with_milvus.ipynb" target="_blank">
    <img src="https://img.shields.io/badge/View%20on%20GitHub-555555?style=flat&logo=github&logoColor=white" alt="GitHub Repository"/>
</a>

# AG2 Multi-Agent RAG with Milvus

This notebook demonstrates how to use [AG2](https://ag2.ai/) (formerly AutoGen) multi-agent conversations with [Milvus](https://milvus.io/) as the vector store for Retrieval-Augmented Generation (RAG).

In this example, two AG2 agents collaborate:
- **Research Agent**: Retrieves relevant documents from Milvus via vector search
- **Analyst Agent**: Synthesizes retrieved information into comprehensive answers

## Prerequisites

Before running this notebook, make sure you have the following dependencies installed:

In [ ]:
! pip install -U "ag2[openai]>=0.11.4,<1.0" pymilvus "pymilvus[milvus_lite]" openai

> If you are using Google Colab, to enable dependencies just installed, you may need to **restart the runtime** (click on the "Runtime" menu at the top of the screen, and select "Restart session" from the dropdown menu).

We will use the models from OpenAI. You should prepare the [api key](https://platform.openai.com/docs/quickstart) `OPENAI_API_KEY` as an environment variable.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-***********"

from openai import OpenAI

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def embed_texts(input_texts):
    """Generate embeddings using OpenAI."""
    response = openai_client.embeddings.create(
        model="text-embedding-3-small", input=input_texts
    )
    return [item.embedding for item in response.data]

## Prepare Sample Data

We'll use a small set of sample documents to demonstrate the RAG pipeline. In production, you would load your own documents (PDFs, web pages, etc.).

In [ ]:
# Sample documents about AI topics
documents = [
    {
        "text": (
            "Retrieval-Augmented Generation (RAG) is a technique that combines "
            "information retrieval with language model generation. It first retrieves "
            "relevant documents from a knowledge base, then uses them as context for "
            "generating accurate, grounded responses. RAG reduces hallucination and "
            "enables models to access up-to-date information beyond their training data."
        ),
        "source": "ai_concepts.md",
    },
    {
        "text": (
            "Vector databases store data as high-dimensional vectors (embeddings) "
            "and enable fast similarity search. Milvus is a leading open-source vector "
            "database that supports billions of vectors with millisecond search latency. "
            "It uses indexing algorithms like HNSW and IVF for efficient approximate "
            "nearest neighbor (ANN) search."
        ),
        "source": "vector_databases.md",
    },
    {
        "text": (
            "Multi-agent systems use multiple AI agents that collaborate to solve "
            "complex tasks. Each agent can have specialized roles, tools, and knowledge. "
            "AG2 (formerly AutoGen) is a popular framework for building multi-agent "
            "conversations where agents can use tools, write code, and coordinate "
            "through structured dialogue patterns."
        ),
        "source": "multi_agent_systems.md",
    },
    {
        "text": (
            "Embedding models convert text into dense numerical vectors that "
            "capture semantic meaning. Similar texts produce similar vectors, enabling "
            "semantic search. Popular embedding models include OpenAI text-embedding-3-small, "
            "sentence-transformers, and BGE models. The choice of embedding model "
            "significantly impacts retrieval quality in RAG systems."
        ),
        "source": "embeddings.md",
    },
    {
        "text": (
            "Chunking is the process of splitting documents into smaller pieces "
            "for embedding and retrieval. Common strategies include fixed-size chunking, "
            "recursive character splitting, and semantic chunking. Optimal chunk size "
            "depends on the use case: 256-512 tokens for precise retrieval, 1000+ tokens "
            "for broader context. Overlap between chunks helps preserve context."
        ),
        "source": "chunking_strategies.md",
    },
]

## Set Up Milvus Vector Store

We'll use **Milvus Lite** which runs entirely in-process — no Docker or server needed.

> - Setting the `uri` as a local file, e.g.`./ag2_milvus_demo.db`, is the most convenient method, as it automatically utilizes [Milvus Lite](https://milvus.io/docs/milvus_lite.md) to store all data in this file.
> - If you have large scale of data, you can set up a more performant Milvus server on [docker or kubernetes](https://milvus.io/docs/quickstart.md). In this setup, please use the server uri, e.g.`http://localhost:19530`, as your `uri`.
> - If you want to use [Zilliz Cloud](https://zilliz.com/cloud), the fully managed cloud service for Milvus, adjust the `uri` and `token`, which correspond to the [Public Endpoint and Api key](https://docs.zilliz.com/docs/on-zilliz-cloud-console#free-cluster-details) in Zilliz Cloud.

In [ ]:
from pymilvus import MilvusClient

# Generate embeddings for all documents
texts = [doc["text"] for doc in documents]
embeddings = embed_texts(texts)
dim = len(embeddings[0])
print(f"Embedding dimension: {dim}")

# Connect to Milvus Lite (in-process, no server needed)
milvus_client = MilvusClient("./ag2_milvus_demo.db")

collection_name = "ag2_rag_collection"

# Drop collection if it exists (for re-running)
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

# Create collection
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=dim,
)

# Insert documents
data = [
    {
        "id": i,
        "vector": embeddings[i],
        "text": documents[i]["text"],
        "source": documents[i]["source"],
    }
    for i in range(len(documents))
]

milvus_client.insert(collection_name=collection_name, data=data)
print(f"Inserted {len(data)} documents into Milvus collection '{collection_name}'")

## Define Milvus Search Function

Create a search function that AG2 agents will use as a tool to retrieve relevant documents from the Milvus collection.

In [ ]:
def search_milvus(query: str, top_k: int = 3) -> str:
    """
    Search Milvus collection for relevant documents.

    Args:
        query: Search query string.
        top_k: Number of results to return.

    Returns:
        Formatted string with retrieved documents and sources.
    """
    # Encode query
    query_embedding = embed_texts([query])

    # Search Milvus
    results = milvus_client.search(
        collection_name=collection_name,
        data=query_embedding,
        limit=top_k,
        output_fields=["text", "source"],
    )

    # Format results
    formatted = []
    for i, hit in enumerate(results[0], 1):
        source = hit["entity"]["source"]
        text = hit["entity"]["text"]
        score = hit["distance"]
        formatted.append(f"[{i}] Source: {source} (similarity: {score:.4f})\n{text}")

    return (
        "\n\n---\n\n".join(formatted) if formatted else "No relevant documents found."
    )


# Test the search function
print(search_milvus("What is RAG?"))

## Set Up AG2 Multi-Agent System

Create AG2 agents that collaborate to answer questions using Milvus retrieval:

1. **Research Agent** — Uses the `search_milvus` tool to find relevant documents
2. **Analyst Agent** — Synthesizes retrieved information into a final answer
3. **User Proxy** — Orchestrates the conversation and executes tool calls

In [ ]:
from autogen import (
    AssistantAgent,
    GroupChat,
    GroupChatManager,
    LLMConfig,
    UserProxyAgent,
)

# AG2 LLM Configuration
llm_config = LLMConfig(
    {
        "model": "gpt-4o-mini",
        "api_key": os.environ["OPENAI_API_KEY"],
        "api_type": "openai",
    }
)

# Create agents
researcher = AssistantAgent(
    name="researcher",
    system_message=(
        "You are a research agent. When asked a question, use the search_documents "
        "tool to retrieve relevant documents from the knowledge base. Present "
        "your findings clearly with source references. If results are insufficient, "
        "try rephrasing your search query."
    ),
    llm_config=llm_config,
)

analyst = AssistantAgent(
    name="analyst",
    system_message=(
        "You are an analyst. Based on the researcher's findings, synthesize the "
        "information into a comprehensive, well-structured answer. Always reference "
        "the source documents. End with TERMINATE when done."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
    is_termination_msg=lambda x: x.get("content", "")
    and "TERMINATE" in x.get("content", ""),
)


# Register Milvus search as AG2 tool
@user_proxy.register_for_execution()
@researcher.register_for_llm(
    description=(
        "Search the Milvus vector database for relevant documents. "
        "Returns document chunks with similarity scores and source references. "
        "Use specific search queries for best results."
    )
)
def search_documents(query: str, top_k: int = 3) -> str:
    """Search Milvus for relevant documents."""
    return search_milvus(query, top_k)

## Run the Multi-Agent Conversation

Now we set up a GroupChat where the agents collaborate, and ask a question that requires searching the knowledge base.

In [ ]:
# Set up GroupChat
group_chat = GroupChat(
    agents=[user_proxy, researcher, analyst],
    messages=[],
    max_round=12,
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

# Run the conversation
user_proxy.run(
    manager,
    message=(
        "What is RAG and how does it work with vector databases? "
        "Also explain how multi-agent systems can improve RAG quality."
    ),
).process()

## Cleanup

Remove the Milvus Lite database file created during this demo.

In [ ]:
import os

if os.path.exists("./ag2_milvus_demo.db"):
    os.remove("./ag2_milvus_demo.db")
    print("Cleaned up Milvus Lite database.")

## Summary

In this tutorial, we built a multi-agent RAG system using AG2 and Milvus. The Research Agent retrieves relevant documents from Milvus via vector search, and the Analyst Agent synthesizes the findings into a comprehensive answer.

### Key Components

| Component | Role |
|-----------|------|
| **Milvus** | Vector storage and similarity search |
| **OpenAI Embeddings** | Text-to-vector conversion |
| **AG2 UserProxy** | Orchestrates conversation, executes tools |
| **AG2 Research Agent** | Calls `search_documents` tool |
| **AG2 Analyst Agent** | Synthesizes findings into final answer |

### Next Steps

- Scale up with more documents and a production Milvus deployment
- Add more specialized agents (fact-checker, summarizer, etc.)
- Use Milvus hybrid search (vector + BM25) for better retrieval
- Add metadata filtering for domain-specific queries